### Projeto 1: Análise de Redes Sociais

Este projeto visa analisar estruturas de redes sociais para fins acadêmicos e construir familiaridade com conceitos de grafos e suas aplicações, utilizando as métricas de grafos para avaliar influências e potenciais de usuários. 

#### Pré-requisitos:
1. Seguir as orientações destacadas no [README.md](README.md)

Para fins de curiosidade, aqui estão informações sobre o dataset anonimizado do Facebook a ser utilizado (https://snap.stanford.edu/data/egonets-Facebook.html). A seguir, estará o "readme-Ego.txt":

```

Files:

nodeId.edges : The edges in the ego network for the node 'nodeId'. Edges are undirected for facebook, and directed (a follows b) for twitter and gplus. The 'ego' node does not appear, but it is assumed that they follow every node id that appears in this file.

nodeId.circles : The set of circles for the ego node. Each line contains one circle, consisting of a series of node ids. The first entry in each line is the name of the circle.

nodeId.feat : The features for each of the nodes that appears in the edge file.

nodeId.egofeat : The features for the ego user.

nodeId.featnames : The names of each of the feature dimensions. Features are '1' if the user has this property in their profile, and '0' otherwise. This file has been anonymized for facebook users, since the names of the features would reveal private data.

```


##### Passos 1 e 2 do projeto: Coletar dados e construir grafo por meio do NetworkX (nx)

In [28]:
import gzip
import random
import pandas as pd
import networkx as nx

#O arquivo já está baixado na pasta artefatos

#Temos que extrair do gzip (gz) 
with gzip.open("artefatos/facebook_combined.txt.gz", "rt") as file:
    Grafo_completo = nx.read_edgelist(file, create_using=nx.Graph(), nodetype=int)

#vendo se deu certo
print(f"Grafo completo - Nós: {Grafo_completo.number_of_nodes()}, Arestas: {Grafo_completo.number_of_edges()}\n")

#Vamos extrair 404 nós aleatoriamente com a biblioteca random (importante notar que isso é pseudo-random, visto que a aleatoriedade é simulada deterministica)
random.seed(74)
nos_da_amostra_404 = random.sample(list(Grafo_completo.nodes), 404)

#Criar o subgrafo induzido de 404 nós
subgrafo_induzido = Grafo_completo.subgraph(nos_da_amostra_404).copy()

print(f"Subgrafo induzido a ser analisado: \nNós: {subgrafo_induzido.number_of_nodes()}, \nArestas: {subgrafo_induzido.number_of_edges()}\n")

#para fins de curiosidade, vamos ver se foi selecionado várias ilhas de conexões ou se o subgrafo está bem conectado:
print(f"Verificação de conexões: {nx.number_connected_components(subgrafo_induzido)}")

Grafo completo - Nós: 4039, Arestas: 88234

Subgrafo induzido a ser analisado: 
Nós: 404, 
Arestas: 846

Verificação de conexões: 81


##### Passo 3 do projeto: Extrair métricas do subgrafo induzido escolhido aleatoriamente

In [29]:
#Vamos começar por extrair as métricas do algoritmo de louvain!

# breve explicação: ============================================================================================================================================
# o algoritmo de louvain é um algoritmo de otimização gulosa (greedy) para extrair comunidades disjuntas de grandes redes. Ele faz isso ao
# agrupar nós e considerar a comunidade deles um nó. Ele faz isso diversas vezes, criando comunidades hierarquicas. O problema é que um nó só pode pertencer
# a 1 comunidade, o que é irrealista na maioria das vezes (por ex: pessoas tem várias comunidades, família, amigos, trabalho...)
# 
# Comunidades são mais comumente chamadas de Clusters em computação. Seus pros são sua rapidez em comparação a outros extratores de comunidades e alta modularidade
# (ou seja, melhores agrupamentos), os seus contras é que já foi demonstrado estatisticamente que o método de Louvain é suscetível a overfit, o que significa que 
# não tem boa generalização para previsões de outros datasets e tende a ficar super-específico. Além disso, também é suscetível a achar "spurious communities" 
#  em subgrafos randômicos (que são pseudo-comunidades, comunidades artificiais que não provêm informação realmente significante para análise), que é nosso caso.
# ==============================================================================================================================================================

#obs: como checamos, existem muitas ilhas de comunidades. Isso interfere nos resultados das métricas. Precisamos utilizar a maior ilha.
# Isso se chama tratamento de conectividade

maior_componente_conexa_nos = max(nx.connected_components(subgrafo_induzido), key=len)
print(f"maiores componentes conexas do subgrafo: {maior_componente_conexa_nos}")

#transformando as maiores componentes (que agora são apenas os nós) em um subgrafo
subgrafo_maiores_componentes_conexas = subgrafo_induzido.subgraph(maior_componente_conexa_nos)

df_metricas = pd.DataFrame(index=subgrafo_maiores_componentes_conexas.nodes())

resultado_louvain = nx.community.louvain_communities(subgrafo_maiores_componentes_conexas)
dicionario_comunidades = {}
for i, comunidade in enumerate(resultado_louvain):
    for no in comunidade:
        dicionario_comunidades[no] = i

df_metricas["clusterizacao de louvain"]= df_metricas.index.map(dicionario_comunidades)

grau_de_centralidade = nx.degree_centrality(subgrafo_maiores_componentes_conexas)
df_metricas["grau de centralidade"] = df_metricas.index.map(grau_de_centralidade)

centralidade_de_intermediacao = nx.betweenness_centrality(subgrafo_maiores_componentes_conexas)
df_metricas["centralidade de intermediacao"] = df_metricas.index.map(centralidade_de_intermediacao)

centralidade_de_proximidade = nx.closeness_centrality(subgrafo_maiores_componentes_conexas)
df_metricas["centralidade de proximidade"] = df_metricas.index.map(centralidade_de_proximidade)

centralidade_de_autovetor = nx.eigenvector_centrality(subgrafo_maiores_componentes_conexas)
df_metricas["centralidade de autovetor"] = df_metricas.index.map(centralidade_de_autovetor)

print(df_metricas)


maiores componentes conexas do subgrafo: {1544, 1034, 1035, 524, 1036, 1553, 1559, 1048, 538, 1052, 541, 544, 545, 1573, 1587, 572, 573, 1086, 1610, 1616, 1107, 1624, 620, 1652, 1142, 1144, 1149, 1151, 1153, 645, 653, 1167, 1680, 1687, 1694, 1698, 1721, 1213, 1738, 1228, 1746, 1752, 1241, 1249, 1255, 1770, 1258, 1772, 1781, 1275, 1284, 1288, 1290, 1292, 1298, 1822, 1312, 1832, 1323, 1325, 1343, 1859, 1355, 1359, 1360, 1881, 348, 1377, 355, 1379, 366, 370, 1397, 1398, 383, 387, 900, 391, 906, 396, 398, 926, 418, 930, 1446, 1448, 1449, 946, 949, 952, 1470, 959, 965, 461, 462, 466, 978, 1491, 477, 991, 481, 486, 1511, 489, 1004, 1005, 1517, 1522, 1523, 1012, 507}
      clusterizacao de louvain  grau de centralidade  \
1544                         5              0.018182   
1034                         5              0.009091   
1035                         2              0.018182   
524                          6              0.072727   
1036                         0              0.05454

##### Passo 4: Produzir um arquivo gephi (.gexf) e salvá-lo em "artefatos"

In [ ]:
#atribuindo as métricas aos nós

dic_louvain = df_metricas["clusterizacao de louvain"].to_dict()
nx.set_node_attributes(subgrafo_maiores_componentes_conexas, dic_louvain, "cluster_louvain")

dic_grau = df_metricas["grau de centralidade"].to_dict()
nx.set_node_attributes(subgrafo_maiores_componentes_conexas, dic_grau, "grau_centralidade")

dic_intermediacao = df_metricas["centralidade de intermediacao"].to_dict()
nx.set_node_attributes(subgrafo_maiores_componentes_conexas, dic_intermediacao, "intermediacao_centralidade")

dic_proximidade = df_metricas["centralidade de proximidade"].to_dict()
nx.set_node_attributes(subgrafo_maiores_componentes_conexas, dic_proximidade, "proximidade_centralidade")

dic_autovetor = df_metricas["centralidade de autovetor"].to_dict()
nx.set_node_attributes(subgrafo_maiores_componentes_conexas, dic_autovetor, "autovetor_centralidade")

#gerando arquivo gephi
nx.write_gexf(subgrafo_maiores_componentes_conexas, "artefatos/subgrafo_404_facebook.gexf")
print(f"Arquivo 'artefatos/subgrafo_404_facebook.gexf' produzido com sucesso e armazenado na pasta './artefatos'")

##### Passo 5: Visualização no Gephi Lite (Browser)

O arquivo foi criado com sucesso. Exporte-o para o [Gephi Lite](https://lite.gephi.org/v1.1.0/) e aplique os devidos filtros. "Basta" interpretar agora!